In [43]:
!pip -q install transformers datasets accelerate evaluate sentencepiece

In [44]:
import os
import random
import warnings
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings("ignore")

In [45]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [47]:
MODEL_NAME = "microsoft/deberta-v3-small"

MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3

SEED = 42

In [48]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

Train Shape: (2000, 8)
Test Shape : (500, 7)


In [49]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [50]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [51]:
train["answer"].value_counts().sort_index()

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

In [52]:
CHOICES = ["A", "B", "C", "D", "E"]

binary_rows = []

for _, row in train.iterrows():

    for choice in CHOICES:

        binary_rows.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "candidate": row[choice],
            "choice": choice,
            "label": int(choice == row["answer"])
        })

binary_train = pd.DataFrame(binary_rows)

print(binary_train.shape)
binary_train.head(10)

(10000, 5)


,id,prompt,candidate,choice,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,0
5,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,A,1
6,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,B,0
7,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,C,0
8,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,D,0
9,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,E,0


In [53]:
binary_train["label"].value_counts()

label
0    8000
1    2000
Name: count, dtype: int64

In [54]:
binary_train[binary_train["id"] == train.iloc[0]["id"]]

,id,prompt,candidate,choice,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,0


In [55]:
train_ids, val_ids = train_test_split(
    train["id"],
    test_size=0.15,
    random_state=42,
    stratify=train["answer"]
)

print(len(train_ids))
print(len(val_ids))

1700
300


In [56]:
train_df = binary_train[
    binary_train["id"].isin(train_ids)
].reset_index(drop=True)


val_df = binary_train[
    binary_train["id"].isin(val_ids)
].reset_index(drop=True)


print(train_df.shape)
print(val_df.shape)

(8500, 5)
(1500, 5)


In [57]:
overlap = set(train_df["id"]) & set(val_df["id"])

print("Common questions:", len(overlap))

Common questions: 0


In [58]:
print(train_df["label"].value_counts())

print()

print(val_df["label"].value_counts())

label
0    6800
1    1700
Name: count, dtype: int64

label
0    1200
1     300
Name: count, dtype: int64


In [59]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(tokenizer)

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [60]:
sample = train_df.iloc[0]

encoding = tokenizer(
    sample["prompt"],
    sample["candidate"],
    max_length=256,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

encoding.keys()

KeysView({'input_ids': tensor([[ 8498,   262,   410,   628,  1330,   294,   458,   269,  2963, 49965,
           280,   268,   866,   277,   262,  1328,   457,   326,   263,   857,
          3861,   302,   880,   262,  2121,  1027,   260,  2963, 49965,  3815,
           272,  3691,  2766,   546,   266,   326, 26535,   272,   269, 11385,
           263,   490,   298,   286,   266,  3034,  1547,   289,   513,   260,
           279,  1328,   264,   262,   695,  3960, 18694,   278,   283,   266,
          2928,  3746,   261,   263,   262,  1328,   264,   262,   723,  3960,
          1512,   266,   447,   272,   296, 12506,  1579,   311,   280,   268,
           451,   326,   260,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,

In [61]:
print(encoding["input_ids"].shape)
print(encoding["attention_mask"].shape)

torch.Size([1, 256])
torch.Size([1, 256])


In [62]:
tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"][0][:30]
)

tokens

['▁Pick',
 '▁the',
 '▁best',
 '▁possible',
 '▁answer',
 ':',
 '▁What',
 '▁is',
 '▁Martin',
 '▁Heidegger',
 "'",
 's',
 '▁view',
 '▁on',
 '▁the',
 '▁relationship',
 '▁between',
 '▁time',
 '▁and',
 '▁human',
 '▁existence',
 '?',
 '▁among',
 '▁the',
 '▁listed',
 '▁options',
 '.',
 '▁Martin',
 '▁Heidegger',
 '▁believes']

In [63]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length):
        
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length


    def __len__(self):

        return len(self.data)


    def __getitem__(self, index):

        row = self.data.iloc[index]

        encoding = self.tokenizer(
            row["prompt"],
            row["candidate"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )


        return {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    row["label"],
                    dtype=torch.long
                )
        }

In [64]:
train_dataset = MCQDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)


val_dataset = MCQDataset(
    val_df,
    tokenizer,
    MAX_LENGTH
)

In [65]:
sample = train_dataset[0]

print(sample.keys())

print(sample["input_ids"].shape)

print(sample["attention_mask"].shape)

print(sample["labels"])

dict_keys(['input_ids', 'attention_mask', 'labels'])
torch.Size([256])
torch.Size([256])
tensor(0)


In [66]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [67]:
batch = next(iter(train_loader))


print(batch["input_ids"].shape)

print(batch["attention_mask"].shape)

print(batch["labels"].shape)

torch.Size([16, 256])
torch.Size([16, 256])
torch.Size([16])


In [68]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    torch_dtype=torch.float32
)

model = model.to(device)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias       

In [69]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() 
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

Total parameters: 141896450
Trainable parameters: 141896450


In [70]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

In [71]:
total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print("Training steps:", total_steps)

Training steps: 1596


In [72]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0


    for batch in loader:

        optimizer.zero_grad()


        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)


        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )


        loss = outputs.loss


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )


        optimizer.step()

        scheduler.step()


        total_loss += loss.item()


    return total_loss / len(loader)

In [73]:
loss = train_epoch(
    model,
    train_loader
)

print("Training loss:", loss)

Training loss: 0.5161248811037469


In [78]:
def get_predictions(model, loader):

    model.eval()

    probabilities = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)


            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


            logits = outputs.logits


            probs = torch.softmax(
                logits,
                dim=1
            )[:,1]


            probabilities.extend(
                probs.cpu().numpy()
            )


    return probabilities

In [79]:
val_probs = get_predictions(
    model,
    val_loader
)


val_result = val_df.copy()

val_result["prob"] = val_probs


val_result.head()

,id,prompt,candidate,choice,label,prob
0,11,Select the most accurate option: What is the P...,The Peierls bracket is a mathematical symbol u...,A,0,0.113016
1,11,Select the most accurate option: What is the P...,The Peierls bracket is a mathematical tool use...,B,0,0.105615
2,11,Select the most accurate option: What is the P...,The Peierls bracket is a Poisson bracket deriv...,C,1,0.124016
3,11,Select the most accurate option: What is the P...,The Peierls bracket is a mathematical symbol u...,D,0,0.115704
4,11,Select the most accurate option: What is the P...,The Peierls bracket is a mathematical tool use...,E,0,0.116597


In [80]:
def map3_score(df):

    scores = []


    for qid, group in df.groupby("id"):


        ranked = group.sort_values(
            "prob",
            ascending=False
        )


        predictions = ranked["choice"].values[:3]


        correct = group[
            group["label"] == 1
        ]["choice"].values[0]


        score = 0


        for i,pred in enumerate(predictions):

            if pred == correct:

                score = 1 / (i+1)
                break


        scores.append(score)


    return np.mean(scores)

In [81]:
score = map3_score(val_result)

print(
    "Validation MAP@3:",
    score
)

Validation MAP@3: 0.7833333333333333


In [82]:
best_score = 0

for epoch in range(2, EPOCHS + 1):

    print(f"\nEpoch {epoch}/{EPOCHS}")

    loss = train_epoch(
        model,
        train_loader
    )

    print(
        "Training Loss:",
        loss
    )


    val_probs = get_predictions(
        model,
        val_loader
    )


    val_result = val_df.copy()

    val_result["prob"] = val_probs


    score = map3_score(
        val_result
    )


    print(
        "Validation MAP@3:",
        score
    )


    if score > best_score:

        best_score = score

        torch.save(
            model.state_dict(),
            "best_deberta.pt"
        )

        print("Saved best model")


Epoch 2/3
Training Loss: 0.4235545122309735
Validation MAP@3: 0.9455555555555555
Saved best model

Epoch 3/3
Training Loss: 0.258891910825737
Validation MAP@3: 0.9666666666666667
Saved best model


In [76]:
# batch = next(iter(train_loader))

# input_ids = batch["input_ids"].to(device)
# attention_mask = batch["attention_mask"].to(device)
# labels = batch["labels"].to(device)

# with torch.no_grad():

#     outputs = model(
#         input_ids=input_ids,
#         attention_mask=attention_mask,
#         labels=labels
#     )


# print(outputs.loss)
# print(outputs.logits[:5])

tensor(0.5508, device='cuda:0')
tensor([[ 0.5507, -0.9106],
        [ 0.6873, -0.5855],
        [ 0.5513, -1.0020],
        [ 0.5731, -0.6646],
        [ 0.6351, -0.5776]], device='cuda:0')


In [77]:
# del model

# torch.cuda.empty_cache()